# Night Vapor Dynamics

In [ ]:
import sympy as sp

# parameters
# Thrust and mass
thr_max = sp.Symbol('thr_max', real=True)  # Maximum thrust [N]

# Mass and geometry
m = sp.Symbol('m', real=True)       # Mass [kg]
XCG = sp.Symbol('XCG', real=True)   # Center of gravity (dimensionless)

# Aerodynamics
S = sp.Symbol('S', real=True)       # Wing area [m^2]
rho = sp.Symbol('rho', real=True)   # Air density [kg/m^3]
g = sp.Symbol('g', real=True)       # Gravity [m/s^2]

# Moments of inertia
Jx = sp.Symbol('Jx', real=True) # Roll moment of inertia [kg·m²]
Jy = sp.Symbol('Jy', real=True) # Pitch moment of inertia [kg·m²]
Jz = sp.Symbol('Jz', real=True) # Yaw moment of inertia [kg·m²]

# Geometry
cbar = sp.Symbol('cbar', real=True) # Mean aerodynamic chord [m]
span = sp.Symbol('span', real=True) # Wingspan [m]

# Control effectiveness (per radian)
Cm0 = sp.Symbol('Cm0', real=True)   # Zero-lift pitching moment
Cldr = sp.Symbol('Cldr', real=True) # Roll due to rudder
Cmde = sp.Symbol('Cmde', real=True) # Pitch due to elevator
Cndr = sp.Symbol('Cndr', real=True) # Yaw due to rudder
CYdr = sp.Symbol('CYdr', real=True) # Side force due to rudder

# Longitudinal stability
CL0 = sp.Symbol('CL0', real=True)       # Lift at zero AoA
CLa = sp.Symbol('CLa', real=True)       # dCL/dα [per rad]
Cma = sp.Symbol('Cma', real=True)       # dCm/dα [per rad]
Cmq = sp.Symbol('Cmq', real=True)       # Pitch damping [per rad/s]
CD0 = sp.Symbol('CD0', real=True)       # Parasite drag
CDCLS = sp.Symbol('CDCLS', real=True)   # Induced drag coefficient

# Lateral-directional stability
Cnb = sp.Symbol('Cnb', real=True)  # Yaw stiffness [per rad]
Clp = sp.Symbol('Clp', real=True)  # Roll damping [per rad/s]
Cnr = sp.Symbol('Cnr', real=True)  # Yaw damping [per rad/s]
Cnp = sp.Symbol('Cnp', real=True)  # Yaw due to roll rate
Clr = sp.Symbol('Clr', real=True)  # Roll due to yaw rate
CYb = sp.Symbol('CYb', real=True)  # Side force due to sideslip [per rad]
CYr = sp.Symbol('CYr', real=True)  # Side force due to yaw rate [per rad/s]
CYp = sp.Symbol('CYp', real=True)  # Side force due to roll rate [per rad/s]

### Parameters

In [ ]:
param_values = {
    thr_max: 0.32,
    m: 0.025,
    S: 0.025,
    rho: 1.225,
    g: 9.81,
    Jx: 1.0e-4,
    Jy: 1.0e-4,
    Jz: 1.0e-4,
    cbar: 0.09,
    span: 0.34,
    Cm0: 0.01,
    Cldr: 0.15,
    Cmde: 0.25,
    Cndr: 0.10,
    CYdr: -0.08,
    CL0: 0.6,
    CLa: 4.8,
    Cma: -0.12,
    Cmq: -0.1,
    CD0: 0.10,
    CDCLS: 0.12,
    Cnb: 0.150,
    Clp: -0.11,
    Cnr: -0.105,
    Cnp: -0.15,
    Clr: 0.10,
    CYb: -0.02,
    CYr: 0.2,
    CYp: 0.1,
}


### State Vector

In [ ]:
# state variables (13)
px, py, pz = sp.symbols('px py pz', real=True)          # position (world)
u, v, w = sp.symbols('u v w', real=True)                # velocity (body)
qw, qx, qy, qz = sp.symbols('qw qx qy qz', real=True)   # quaternion (world to body)
p, q, r = sp.symbols('p q r', real=True)                # angular velocity (body)

# state vector
X = sp.Matrix([px, py, pz, u, v, w, qw, qx, qy, qz, p, q, r])

### Control Vector

In [ ]:
# input variables (3)
thrust, elevation, rudder = sp.symbols('thrust elevation rudder', real=True)
U = sp.Matrix([thrust, elevation, rudder])

### Airspeed & Angles

In [ ]:
V = sp.sqrt(u**2 + v**2 + w**2)  # airspeed

# epsilon for division by zero
V_safe = sp.Max(V, 1e-6)
u_safe = sp.Max(sp.Abs(u), 1e-6) * sp.sign(u)  # Safe u velocity component

alpha = sp.atan2(-w, u_safe) # angle of attack
beta = sp.asin(v / V_safe)   # sideslip angle

qbar = 0.5 * rho * V_safe**2 # dynamic pressure

### Aerodynamic Coefficients

In [ ]:
# Lift
CL = CL0 + (CLa * alpha)

# Drag
CD = CD0 + (CDCLS * CL**2)

# Side force
CY = -(CYb * beta) + (CYdr * rudder) + ((span / (2 * V_safe)) * ((CYp * p) + (CYr * r)))

### Rotation Matrix (R_nb)

In [ ]:
# Rotation matrix from quaternion
def R_from_quat(qw, qx, qy, qz):
    R11 = 1 - 2*(qy**2 + qz**2)
    R12 = 2*(qx*qy - qw*qz)
    R13 = 2*(qx*qz + qw*qy)
    R21 = 2*(qx*qy + qw*qz)
    R22 = 1 - 2*(qx**2 + qz**2)
    R23 = 2*(qy*qz - qw*qx)
    R31 = 2*(qx*qz - qw*qy)
    R32 = 2*(qy*qz + qw*qx)
    R33 = 1 - 2*(qx**2 + qy**2)
    return sp.Matrix([[R11, R12, R13],
                      [R21, R22, R23],
                      [R31, R32, R33]])

# Make quaternion (wind -> body)
cos_half_alpha = sp.cos(alpha/2)
sin_half_alpha = sp.sin(alpha/2)
cos_half_beta = sp.cos(beta/2)
sin_half_beta = sp.sin(beta/2)

# q_bn (body <- wind)
qw_bn = cos_half_beta * cos_half_alpha
qx_bn = cos_half_beta * sin_half_alpha
qy_bn = sin_half_beta * cos_half_alpha
qz_bn = -sin_half_beta * sin_half_alpha

# q_nb (wind -> body)
qw_nb = qw_bn
qx_nb = -qx_bn
qy_nb = -qy_bn
qz_nb = -qz_bn

# Rotation matrix (R_nb: wind -> body)
R_nb = R_from_quat(qw_nb, qx_nb, qy_nb, qz_nb)

### Aerodynamic force

In [ ]:
Dw = qbar * S * CD  # Drag force
Lw = qbar * S * CL  # Lift force
Yw = qbar * S * CY  # Side force

F_wind = sp.Matrix([-Dw, Yw, Lw])
F_aero_b = R_nb * F_wind

# Thrust force
T_b = sp.Matrix([thr_max*thrust, 0, 0])

# Gravity
R_wb = R_from_quat(qw, qx, qy, qz)
W_b = R_wb * sp.Matrix([0, 0, -m*g])

# Total force in body frame
F_b = F_aero_b + T_b + W_b

### Total Moment

In [ ]:
# Rolling moment
Cl = -(Cldr * rudder) + (Clp * (span / (2 * V_safe)) * p) + (Clr * (span / (2 * V_safe)) * r)
# Pitching moment
Cm = Cm0 + (Cma * alpha) + (Cmde * elevation) + (Cmq * (cbar / (2 * V_safe)) * q)
# Yawing moment
Cn = (Cnb * beta) + (Cndr * rudder) + (Cnp * (span / (2 * V_safe)) * p) + (Cnr * (span / (2 * V_safe)) * r)

# Aerodynamic moments (body frame)
Mx = qbar * S * span * Cl
My = qbar * S * cbar * Cm
Mz = qbar * S * span * Cn

M_b = sp.Matrix([Mx, My, Mz])

### Helper Functions

In [ ]:
def skew(ax, ay, az):
    return sp.Matrix([[0, -az,  ay],
                      [az,  0, -ax],
                      [-ay, ax,  0]])

def Omega(p, q, r):
    return sp.Matrix([
        [0,  -p, -q, -r],
        [p,   0,  r, -q],
        [q,  -r,  0,  p],
        [r,   q, -p,  0],
    ])

### Continuous-time Dynamics f(X,U)

In [ ]:
# Quaternion kinematics
R_bw = R_wb.T
v_b = sp.Matrix([u, v, w])
p_dot = R_bw * v_b

# Translational dynamics
omega_b = sp.Matrix([p, q, r])
v_b_dot = (1/m) * F_b - skew(p, q, r) * v_b

# Quaternion kinematics
q_wb = sp.Matrix([qw, qx, qy, qz])
q_wb_dot = (1/2) * Omega(p, q, r) * q_wb

# Rotational dynamics
J = sp.diag(Jx, Jy, Jz)
J_w = J * omega_b
omega_b_dot = J.LUsolve(M_b - skew(p, q, r) * J_w)

# continuous-time dynamics
f = sp.Matrix.vstack(p_dot, v_b_dot, q_wb_dot, omega_b_dot)

### Jacobian

In [ ]:
F = f.jacobian(X)  # Jacobian w.r.t. states
G = f.jacobian(U)  # Jacobian w.r.t. inputs